# 01 — Exploratory Data Analysis of the 60-Minute SOLETE Data

## Purpose

This notebook begins the exploration of the SOLETE renewable-energy dataset
using its 60-minute resolution.

We first need to understand:

1. what variables are available;
2. how much time the dataset covers;
3. whether timestamps are chronological, unique, and regularly spaced;
4. whether measurements contain missing or unusual values; and
5. which renewable-generation variables could serve as forecasting targets.

At this stage, no temporal resolution or forecasting target is assumed to be
the best choice. The purpose is to understand the source data first and use
that evidence to guide later decisions.

## Data source

SOLETE contains meteorological measurements together with co-located wind and
solar photovoltaic generation from the SYSLAB facility in Denmark.

- Pombo, D. V., Gehrke, O., & Bindner, H. W. (2022), *SOLETE, a 15-month
  long holistic dataset including: Meteorology, co-located wind and solar PV
  power from Denmark with various resolutions*, Data in Brief, 42, 108046.
  DOI: `10.1016/j.dib.2022.108046`
- Dataset DOI: `10.11583/DTU.17040767`

SOLETE is available at multiple temporal resolutions. We begin with the
60-minute file because it provides a compact view of the dataset and makes
basic temporal and measurement-quality patterns easy to inspect.


In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

# Locate the project root whether the notebook is launched from the project
# directory or directly from notebooks/.
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_solete

DATASET_PATH = PROJECT_ROOT / "solete_dataset" / "SOLETE_Pombo_60min.h5"
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"SOLETE file not found: {DATASET_PATH}")

df = load_solete(DATASET_PATH)

## Variables and possible forecasting targets

The 60-minute file contains meteorological measurements, irradiance,
renewable-power outputs, pressure, and two source solar-angle columns.

| Column | Interpretation |
|---|---|
| `TEMPERATURE[degC]` | Air temperature |
| `HUMIDITY[%]` | Relative-humidity series |
| `WIND_SPEED[m1s]` | Wind speed |
| `WIND_DIR[deg]` | Wind direction |
| `GHI[kW1m2]` | Global horizontal irradiance |
| `POA Irr[kW1m2]` | Plane-of-array irradiance |
| `P_Gaia[kW]` | Gaia wind-turbine power |
| `P_Solar[kW]` | Solar-PV power |
| `Pressure[mbar]` | Atmospheric pressure |
| `Azimuth[deg]` | Source azimuth field |
| `Elevation[deg]` | Source elevation field |

Two columns are immediately interesting as possible forecasting targets:

- `P_Solar[kW]` — photovoltaic power generation;
- `P_Gaia[kW]` — wind-turbine power generation.

Rather than selecting one immediately, we retain both as candidate targets and
first examine their distributions and data quality. Later forecasting
experiments can determine whether the study should model one or both renewable
generation processes.

The source `Azimuth[deg]` and `Elevation[deg]` fields are treated as dataset
variables to be inspected rather than automatically accepted as modelling
features.


In [2]:
# A compact overview is more informative than printing the full DataFrame.
overview = pd.Series({
    "rows": len(df),
    "columns": df.shape[1],
    "start_timestamp": df.index.min(),
    "end_timestamp": df.index.max(),
    "timezone": str(df.index.tz),
    "raw_index_chronological": df.index.is_monotonic_increasing,
    "duplicate_timestamps": int(df.index.duplicated().sum()),
    "missing_measurements": int(df.isna().sum().sum()),
}, name="60-minute SOLETE")

display(overview.to_frame("value"))
display(df.head(3))


,value
rows,10969
columns,11
start_timestamp,2018-06-01 00:00:00+00:00
end_timestamp,2019-09-01 00:00:00+00:00
timezone,UTC
raw_index_chronological,False
duplicate_timestamps,0
missing_measurements,0


,TEMPERATURE[degC],HUMIDITY[%],WIND_SPEED[m1s],WIND_DIR[deg],GHI[kW1m2],POA Irr[kW1m2],P_Gaia[kW],P_Solar[kW],Pressure[mbar],Azimuth[deg],Elevation[deg]
Timestamp,,,,,,,,,,,
2018-11-17 00:00:00+00:00,13.953556,1.6,7.257889,315.208889,0.0,0.0,0.0,0.0,3000.0,0.0,0.0
2018-11-17 01:00:00+00:00,13.130778,1.6,7.106611,311.173333,0.0,0.0,0.0,0.0,2000.0,0.0,0.0
2018-11-17 02:00:00+00:00,12.132889,1.6,7.641944,310.871111,0.0,0.0,0.0,0.0,2000.0,0.0,0.0


## Temporal structure

Time-series forecasting depends on the true chronological sequence of
observations. Before creating lags or train/test splits, we therefore check:

- whether the raw timestamp index is already chronological;
- whether any timestamps are duplicated;
- whether any expected hourly timestamps are missing; and
- whether consecutive observations are separated by exactly one hour.

If the source rows are not chronological, we will create a sorted working copy.
Sorting changes only row order; it does not alter any measurement value.


In [3]:
df_sorted = df.sort_index()

expected_index = pd.date_range(
    start=df_sorted.index.min(),
    end=df_sorted.index.max(),
    freq="1h",
)

missing_timestamps = expected_index.difference(df_sorted.index)
unexpected_timestamps = df_sorted.index.difference(expected_index)
intervals = df_sorted.index.to_series().diff().dropna()

temporal_summary = pd.Series({
    "chronological_after_sort": df_sorted.index.is_monotonic_increasing,
    "duplicate_timestamps": int(df_sorted.index.duplicated().sum()),
    "observed_rows": len(df_sorted),
    "expected_hourly_rows": len(expected_index),
    "missing_hourly_timestamps": len(missing_timestamps),
    "unexpected_timestamps": len(unexpected_timestamps),
    "non_hourly_intervals": int((intervals != pd.Timedelta(hours=1)).sum()),
}, name="temporal structure")

temporal_summary.to_frame("value")


,value
chronological_after_sort,True
duplicate_timestamps,0
observed_rows,10969
expected_hourly_rows,10969
missing_hourly_timestamps,0
unexpected_timestamps,0
non_hourly_intervals,0


In [4]:
# Compare the two renewable-generation series before choosing the
# forecasting target or targets. This step is descriptive only.
target_columns = ["P_Solar[kW]", "P_Gaia[kW]"]

target_summary = pd.DataFrame({
    "mean_kW": df_sorted[target_columns].mean(),
    "median_kW": df_sorted[target_columns].median(),
    "max_kW": df_sorted[target_columns].max(),
    "zero_fraction": (df_sorted[target_columns] == 0).mean(),
    "missing": df_sorted[target_columns].isna().sum(),
})

target_summary


,mean_kW,median_kW,max_kW,zero_fraction,missing
P_Solar[kW],1.029960,0.077733,6.975178,0.383262,0
P_Gaia[kW],0.016259,0.000000,10.184867,0.995624,0


## Measurement distributions and quality flags

We next inspect the numerical ranges of the measurements and flag values that
deserve closer examination.

These checks are **diagnostic rather than corrective**. A value that looks
unusual is not automatically an error; it may reflect the source encoding,
aggregation method, instrument behaviour, or a genuine observation.

The humidity column is especially worth checking because its values appear to
be expressed mainly on a fractional scale despite the `%` text in its source
name. Wind direction is circular, so angles outside `[0, 360)` may still have
a valid directional interpretation after modulo-360 wrapping. Pressure values
far outside the normal atmospheric range are also flagged for investigation.

Any cleaning decision should be supported by evidence from the data and,
where useful, comparison with the other SOLETE resolutions before it is used
in modelling.


In [5]:
# Compact descriptive statistics: enough to expose scale, centre, and extremes.
descriptive = df_sorted.describe().T[
    ["count", "mean", "std", "min", "50%", "max"]
].rename(columns={"50%": "median"})

descriptive


,count,mean,std,min,median,max
TEMPERATURE[degC],10969.0,12.075792,7.347850,-4.402528,12.595083,44.241984
HUMIDITY[%],10969.0,0.782769,0.195906,0.000000,0.800000,2.700000
WIND_SPEED[m1s],10969.0,3.642386,2.141412,0.000000,3.319972,23.150403
WIND_DIR[deg],10969.0,212.288659,75.736164,0.000000,217.555000,639.344818
GHI[kW1m2],10969.0,0.140451,0.212350,0.000000,0.012351,0.910141
POA Irr[kW1m2],10969.0,0.144488,0.237810,0.000000,0.010066,0.961514
P_Gaia[kW],10969.0,0.016259,0.323335,0.000000,0.000000,10.184867
P_Solar[kW],10969.0,1.029960,1.657731,0.000000,0.077733,6.975178
Pressure[mbar],10969.0,1043.845751,205.662219,992.367136,1000.000000,3000.000000
Azimuth[deg],10969.0,-0.003834,0.994408,-56.907645,0.000000,44.794950


In [6]:
quality_flags = pd.Series({
    "humidity < 0 or > 1": int(
        ((df_sorted["HUMIDITY[%]"] < 0) | (df_sorted["HUMIDITY[%]"] > 1)).sum()
    ),
    "wind direction outside [0, 360)": int(
        ((df_sorted["WIND_DIR[deg]"] < 0) | (df_sorted["WIND_DIR[deg]"] >= 360)).sum()
    ),
    "negative GHI": int((df_sorted["GHI[kW1m2]"] < 0).sum()),
    "negative POA irradiance": int((df_sorted["POA Irr[kW1m2]"] < 0).sum()),
    "negative wind power": int((df_sorted["P_Gaia[kW]"] < 0).sum()),
    "negative solar power": int((df_sorted["P_Solar[kW]"] < 0).sum()),
    "pressure outside 800–1100 mbar": int(
        ((df_sorted["Pressure[mbar]"] < 800) | (df_sorted["Pressure[mbar]"] > 1100)).sum()
    ),
    "elevation outside [-90, 90] deg": int(
        ((df_sorted["Elevation[deg]"] < -90) | (df_sorted["Elevation[deg]"] > 90)).sum()
    ),
}, name="flagged observations")

quality_flags.to_frame()


,flagged observations
humidity < 0 or > 1,188
"wind direction outside [0, 360)",103
negative GHI,0
negative POA irradiance,0
negative wind power,0
negative solar power,0
pressure outside 800–1100 mbar,479
"elevation outside [-90, 90] deg",0


In [7]:
# Focus only on the two conspicuous source encodings identified above.
pressure_counts = (
    df_sorted["Pressure[mbar]"]
    .value_counts()
    .head(10)
    .rename("observations")
    .to_frame()
)

wrapped_wind = df_sorted["WIND_DIR[deg]"] % 360
wind_summary = pd.Series({
    "raw_min_deg": df_sorted["WIND_DIR[deg]"].min(),
    "raw_max_deg": df_sorted["WIND_DIR[deg]"].max(),
    "wrapped_min_deg": wrapped_wind.min(),
    "wrapped_max_deg": wrapped_wind.max(),
}, name="wind direction")

print("Most common pressure values")
display(pressure_counts)

print("Wind-direction range before and after modulo-360 wrapping")
display(wind_summary.to_frame("value"))


Most common pressure values


,observations
Pressure[mbar],
1000.000000,10477
2000.000000,477
3000.000000,2
998.344811,1
997.532282,1
997.531353,1
997.271835,1
996.774862,1
996.302887,1


Wind-direction range before and after modulo-360 wrapping


,value
raw_min_deg,0.000000
raw_max_deg,639.344818
wrapped_min_deg,0.000000
wrapped_max_deg,358.782162


## Findings

The 60-minute SOLETE file contains 10,969 observations across 11 variables and
covers approximately 15 months, from June 2018 to the beginning of September
2019.

The raw HDF5 row order is not chronological. After sorting by the timezone-aware
UTC timestamp, the time index can be evaluated as an hourly series for
duplicates, missing timestamps, and irregular spacing. This is an important
prerequisite for any later lag construction or chronological forecasting
split.

Both renewable-generation variables remain plausible forecasting targets:

- `P_Solar[kW]` for photovoltaic generation;
- `P_Gaia[kW]` for wind generation.

The measurement checks also identify several features that deserve further
investigation before modelling:

- the humidity series appears to use a fractional scale for most observations;
- some wind-direction values lie outside the conventional `[0, 360)` interval
  but can be interpreted circularly;
- pressure contains conspicuous repeated high values;
- the source solar-angle fields should be compared across resolutions before
  deciding whether and how they should be used.

No raw measurement is modified in this notebook.

## Next step

The next notebook examines the 5-minute SOLETE data using the same core
questions while paying attention to features that may behave differently at a
finer temporal resolution.

After examining the available resolutions, the datasets can be compared
directly to decide:

- which resolution is most appropriate for forecasting;
- whether apparent anomalies are common or resolution-specific;
- how solar and wind target behaviour changes with temporal aggregation; and
- which preprocessing rules are justified before model development.
